# Gradients and How Models Actually Learn

Companion notebook for Act 1, Article 3 of the ML series.
Reproduces Article 1/2's cleaning pipeline on the real TLC data, then
implements loss functions and gradient descent (batch / SGD / mini-batch)
entirely from scratch — no `model.fit()` anywhere in this notebook.

Article: *(add link when published)*

In [ ]:
import numpy as np
import pandas as pd

# Pinned URL — same file Articles 1 & 2 used
DATA_URL = "https://d37ci6vzurychx.cloudfront.net/trip-data/yellow_tripdata_2024-01.parquet"

RNG_SEED = 0
SAMPLE_SIZE = 5000

# The from-scratch GD demos below use a 4-feature subset — small enough
# that batch GD, SGD, and mini-batch all converge in seconds, and small
# enough to print a closed-form solution to compare against.
DEMO_FEATURES = ["trip_distance", "fare_amount", "fare_per_mile", "is_rush_hour"]
TARGET = "tip_pct"

## 1. Reproduce Article 1/2 pipeline, then sample down to the demo subset

In [ ]:
def clean_trips(df):
    df = df[df["fare_amount"] > 0]
    df = df[df["tip_amount"] >= 0]
    df = df[df["passenger_count"] >= 1]
    df = df[df["passenger_count"] <= 5]
    df = df[df["trip_distance"] > 0]
    df = df[df["trip_distance"] <= 50]
    df = df[df["fare_amount"] >= 3.0]
    df = df[df["fare_amount"] <= 200]
    df["trip_duration_min"] = (
        df["tpep_dropoff_datetime"] - df["tpep_pickup_datetime"]
    ).dt.total_seconds() / 60
    df = df[df["trip_duration_min"] > 1]
    df = df[df["trip_duration_min"] <= 120]
    df = df[df["RatecodeID"].isin([1, 2])]
    return df.reset_index(drop=True)

def engineer_features(df):
    dt = df["tpep_pickup_datetime"]
    df["is_rush_hour"] = (
        ((dt.dt.hour >= 7) & (dt.dt.hour <= 9)) |
        ((dt.dt.hour >= 16) & (dt.dt.hour <= 19))
    ).astype(int)
    df["fare_per_mile"]   = (df["fare_amount"] / df["trip_distance"]).clip(upper=50)
    df["is_card_payment"] = (df["payment_type"] == 1).astype(int)
    return df

print("Loading TLC data …")
df = pd.read_parquet(DATA_URL)
df = clean_trips(df)
df = engineer_features(df)
df = df[df["is_card_payment"] == 1].copy()
df[TARGET] = (df["tip_amount"] / df["fare_amount"] * 100).clip(0, 50)
df = df[DEMO_FEATURES + [TARGET]].dropna()
df = df.sample(n=SAMPLE_SIZE, random_state=RNG_SEED).reset_index(drop=True)
print(f"Demo dataset: {len(df):,} real trips, features={DEMO_FEATURES}")

`fare_amount` and `fare_per_mile` are raw dollar values — up to \$200 and
\$50/mile respectively. That's a much larger scale than a synthetic,
pre-normalized feature, and feeding it straight into gradient descent blows
up at every learning rate this notebook tests. Standardize first, same
role `StandardScaler` plays in Article 2's sklearn pipelines.

In [ ]:
X = df[DEMO_FEATURES].to_numpy(dtype=float)
y = df[TARGET].to_numpy(dtype=float)

X_mean, X_std = X.mean(axis=0), X.std(axis=0)
X = (X - X_mean) / X_std

print("Feature means (raw):", np.round(X_mean, 3))
print("Feature stds  (raw):", np.round(X_std, 3))

def add_bias(X):
    return np.hstack([np.ones((len(X), 1)), X])

def predict(Xb, w):
    return Xb @ w

Xb = add_bias(X)

## 2. Loss functions and their gradients

In [ ]:
def mse_loss(y, pred):
    return float(np.mean((pred - y) ** 2))

def gradient_mse(Xb, y, w):
    pred = predict(Xb, w)
    return (2 / len(y)) * Xb.T @ (pred - y)

def gradient_mae(Xb, y, w):
    pred = predict(Xb, w)
    return (1 / len(y)) * Xb.T @ np.sign(pred - y)

def gradient_huber(Xb, y, w, delta=5.0):
    pred = predict(Xb, w)
    err = pred - y
    grad = np.where(np.abs(err) <= delta, err, delta * np.sign(err))
    return (1 / len(y)) * Xb.T @ grad

def run_gd(grad_fn, Xb, y, lr, epochs):
    w = np.zeros(Xb.shape[1])
    for _ in range(epochs):
        w -= lr * grad_fn(Xb, y, w)
    return w

## 3. Loss functions vs. injected outliers

Inject 15 garbage `tip_pct` values (set to 200 — an impossible tip
percentage, like a data-entry error) into the 5,000-row sample.

An earlier version of this experiment compared MSE / MAE / Huber by RMSE
on the clean holdout. That's a biased comparison: RMSE is literally the
quantity MSE minimizes, so judging MAE- or Huber-trained weights by RMSE
punishes them for not optimizing the metric they were never trying to
optimize. The metric-independent question is: fit each loss on clean-only
data, fit it again with the outliers mixed in, and measure how far the
learned weights moved.

In [ ]:
rng = np.random.default_rng(RNG_SEED)
outlier_idx = rng.choice(len(y), 15, replace=False)
y_out = y.copy()
y_out[outlier_idx] = 200  # garbage tip_pct, like a data-entry error

clean_mask = np.ones(len(y), dtype=bool)
clean_mask[outlier_idx] = False
Xb_clean, y_clean = Xb[clean_mask], y[clean_mask]

# epochs chosen so every loss has actually converged (verified by checking
# loss/weights stop moving at 2x these epoch counts — MAE's constant ±1
# subgradient converges far slower than MSE's smooth gradient)
configs = {
    "MSE":   (gradient_mse,   0.1, 5000),
    "MAE":   (gradient_mae,   0.1, 50000),
    "Huber": (gradient_huber, 0.1, 20000),
}

for name, (grad_fn, lr, epochs) in configs.items():
    w_clean = run_gd(grad_fn, Xb_clean, y_clean, lr=lr, epochs=epochs)
    w_out   = run_gd(grad_fn, Xb,       y_out,   lr=lr, epochs=epochs)
    drift = float(np.linalg.norm(w_out - w_clean))
    print(f"{name:6s} w_clean={np.round(w_clean, 3)}")
    print(f"{'':6s} w_out  ={np.round(w_out, 3)}   weight drift (L2): {drift:.4f}")

## 4. Batch vs. SGD vs. mini-batch gradient descent

All three use MSE and should converge toward the same closed-form
(`np.linalg.lstsq`) optimum. What differs is the path and the cost of
getting there.

In [ ]:
def batch_gd(Xb, y, lr=0.1, epochs=200):
    w = np.zeros(Xb.shape[1])
    losses = []
    for _ in range(epochs):
        w -= lr * gradient_mse(Xb, y, w)
        losses.append(mse_loss(y, predict(Xb, w)))
    return w, losses

def sgd(Xb, y, lr=0.01, epochs=10, seed=0):
    rng = np.random.default_rng(seed)
    w = np.zeros(Xb.shape[1])
    losses = []
    for _ in range(epochs):
        for i in rng.permutation(len(y)):
            xi, yi = Xb[i:i+1], y[i:i+1]
            grad = 2 * xi.T @ (predict(xi, w) - yi)
            w -= lr * grad.flatten()
        losses.append(mse_loss(y, predict(Xb, w)))
    return w, losses

def minibatch_gd(Xb, y, lr=0.05, epochs=50, batch_size=64, seed=0):
    rng = np.random.default_rng(seed)
    w = np.zeros(Xb.shape[1])
    losses = []
    for _ in range(epochs):
        idx = rng.permutation(len(y))
        for start in range(0, len(y), batch_size):
            batch = idx[start:start+batch_size]
            grad = gradient_mse(Xb[batch], y[batch], w)
            w -= lr * grad
        losses.append(mse_loss(y, predict(Xb, w)))
    return w, losses

In [ ]:
w_star, *_ = np.linalg.lstsq(Xb, y, rcond=None)
print("Closed-form weights (the actual optimum):", np.round(w_star, 3))

w_batch, loss_batch = batch_gd(Xb, y, lr=0.1, epochs=200)
w_sgd,   loss_sgd   = sgd(Xb, y, lr=0.01, epochs=10, seed=RNG_SEED)
w_mb,    loss_mb    = minibatch_gd(Xb, y, lr=0.05, epochs=50, batch_size=64, seed=RNG_SEED)

print("Batch GD final weights:  ", np.round(w_batch, 3), "final loss:", round(loss_batch[-1], 3))
print("SGD final weights:       ", np.round(w_sgd, 3),   "final loss:", round(loss_sgd[-1], 3))
print("Mini-batch final weights:", np.round(w_mb, 3),    "final loss:", round(loss_mb[-1], 3))

print("\nBatch GD loss (every 40 of 200 epochs): ", [round(v,3) for v in loss_batch[::40]])
print("SGD loss (every 1 of 10 epochs):         ", [round(v,3) for v in loss_sgd])
print("Mini-batch loss (every 10 of 50 epochs): ", [round(v,3) for v in loss_mb[::10]])

## 5. The learning rate sweep

`lr` scales the step size. Get it wrong and nothing above matters —
including the deceptive case where a too-large learning rate looks
*better* than a well-tuned one for the first few epochs, then diverges.

In [ ]:
for lr in [0.001, 0.01, 0.1, 0.5, 1.0, 1.5]:
    _, losses = batch_gd(Xb, y, lr, epochs=30)
    e5, e30 = losses[4], losses[-1]
    print(f"lr={lr:<6} loss@epoch5={e5:>16.3f}   loss@epoch30={e30:.3f}"
          if np.isfinite(e30) else f"lr={lr:<6} loss@epoch5={e5:>16.3f}   loss@epoch30=inf")

In [ ]:
# Zoom in on epochs 1–6 for lr=0.1 vs lr=0.5 — where the crossover happens
for lr in [0.1, 0.5]:
    _, losses = batch_gd(Xb, y, lr, epochs=6)
    print(f"lr={lr}:", [round(v, 3) for v in losses])

## 6. Visualizing the loss surface and the learning-rate trap

Two real visuals, saved to `output/`:

1. The loss surface batch GD actually descends, sliced to the two
   correlated weights from Section 4 (`trip_distance`, `fare_amount`),
   with the real 200-step batch GD path drawn on top.
2. The same five learning rates from Section 5, zoomed to the first 10
   epochs (where `lr=0.5` briefly wins) next to the full 30-epoch blowup.

In [ ]:
import os
import matplotlib.pyplot as plt
from matplotlib.colors import LinearSegmentedColormap

OUTPUT_DIR = "output"
os.makedirs(OUTPUT_DIR, exist_ok=True)

BLUE, ORANGE, GREEN, RED = "#2980b9", "#e67e22", "#27ae60", "#e34948"
BLUE_SEQ = ["#cde2fb", "#9ec5f4", "#6da7ec", "#3987e5",
            "#2a78d6", "#1c5cab", "#104281", "#0d366b"]

def batch_gd_path(Xb, y, lr=0.1, epochs=200):
    w = np.zeros(Xb.shape[1])
    history = [w.copy()]
    for _ in range(epochs):
        w = w - lr * gradient_mse(Xb, y, w)
        history.append(w.copy())
    return np.array(history)

def loss_surface(Xb, y, w_fixed, i, j, w1_range, w2_range):
    # MSE is an exact quadratic in the weights for linear regression, so
    # the surface is computed analytically rather than by brute force.
    Xi, Xj = Xb[:, i], Xb[:, j]
    base = Xb @ w_fixed - Xi * w_fixed[i] - Xj * w_fixed[j]
    r0 = base - y
    c0, c1, c2 = np.mean(r0**2), 2*np.mean(r0*Xi), 2*np.mean(r0*Xj)
    c11, c22, c12 = np.mean(Xi**2), np.mean(Xj**2), 2*np.mean(Xi*Xj)
    W1, W2 = np.meshgrid(w1_range, w2_range)
    Loss = c0 + c1*W1 + c2*W2 + c11*W1**2 + c22*W2**2 + c12*W1*W2
    return W1, W2, Loss

In [ ]:
i, j = 1, 2  # trip_distance weight, fare_amount weight
path = batch_gd_path(Xb, y, lr=0.1, epochs=200)

pad = 2.5
w1_lo, w1_hi = min(0, w_star[i]) - pad, max(0, w_star[i]) + pad
w2_lo, w2_hi = min(0, w_star[j]) - pad, max(0, w_star[j]) + pad
w1_range = np.linspace(w1_lo, w1_hi, 200)
w2_range = np.linspace(w2_lo, w2_hi, 200)
W1, W2, Loss = loss_surface(Xb, y, w_star, i, j, w1_range, w2_range)

cmap = LinearSegmentedColormap.from_list("blue_seq", BLUE_SEQ)
fig, ax = plt.subplots(figsize=(8, 6.5))
cf = ax.contourf(W1, W2, Loss, levels=30, cmap=cmap)
lines = ax.contour(W1, W2, Loss, levels=8, colors="white", linewidths=0.6, alpha=0.6)
ax.clabel(lines, inline=True, fontsize=7, fmt="%.0f")
fig.colorbar(cf, ax=ax).set_label("MSE loss (higher = worse)", color="#52514e")

ax.plot(path[:, i], path[:, j], color=ORANGE, lw=2, zorder=4, label="Batch GD path (200 steps)")
ax.scatter(path[0, i], path[0, j], color=ORANGE, edgecolor="white", s=50, zorder=5, marker="s", label="Start (w = 0)")
ax.scatter(w_star[i], w_star[j], color="white", edgecolor="black", s=140, zorder=5, marker="*", label="Closed-form optimum")
ax.set_xlabel("trip_distance weight")
ax.set_ylabel("fare_amount weight")
ax.set_title("The loss surface batch GD descends\none convex valley — no separate local minima or maxima to fall into", fontsize=11)
ax.legend(loc="upper left", frameon=True, fontsize=9)
plt.tight_layout()
plt.savefig(f"{OUTPUT_DIR}/loss_landscape.png", dpi=150)
plt.show()

In [ ]:
lr_colors = [(0.001, BLUE_SEQ[1]), (0.01, BLUE_SEQ[3]), (0.1, BLUE), (0.5, ORANGE), (1.0, RED)]
all_losses = {lr: batch_gd(Xb, y, lr, epochs=30)[1] for lr, _ in lr_colors}

fig, (ax_zoom, ax_full) = plt.subplots(1, 2, figsize=(13, 6))

for lr, color in lr_colors:
    if lr == 1.0:
        continue
    losses = all_losses[lr][:10]
    epochs = np.arange(1, len(losses) + 1)
    ax_zoom.plot(epochs, losses, color=color, lw=2.2, marker="o", markersize=4)
    ax_zoom.annotate(f"lr={lr}", xy=(epochs[-1], losses[-1]), xytext=(6, 0),
                      textcoords="offset points", va="center", fontsize=9, color=color, fontweight="bold")
ax_zoom.axvline(5, color="#898781", lw=1, linestyle=":")
ax_zoom.text(5.15, 0.95, "epoch 5\ncrossover", transform=ax_zoom.get_xaxis_transform(),
             ha="left", va="top", fontsize=8, color="#52514e")
ax_zoom.set_yscale("log")
ax_zoom.set_xlabel("epoch")
ax_zoom.set_ylabel("MSE loss (log scale)")
ax_zoom.set_title("First 10 epochs — lr=0.5 leads, then reverses", fontsize=10)
ax_zoom.grid(True, which="both", axis="y", color="#e1e0d9", linewidth=0.6)
ax_zoom.set_axisbelow(True)

for lr, color in lr_colors:
    losses = all_losses[lr]
    epochs = np.arange(1, len(losses) + 1)
    ax_full.plot(epochs, losses, color=color, lw=2.2, marker="o", markersize=3)
    if lr in (0.5, 1.0):
        ax_full.annotate(f"lr={lr}", xy=(epochs[-1], losses[-1]), xytext=(6, 0),
                          textcoords="offset points", va="center", fontsize=9, color=color, fontweight="bold")
ax_full.annotate("lr = 0.001 / 0.01 / 0.1\n(converge, barely visible at this scale)",
                  xy=(4, all_losses[0.1][3]), xytext=(15, 55), textcoords="offset points",
                  fontsize=8, color="#52514e", arrowprops=dict(arrowstyle="-", color="#898781", lw=0.8))
ax_full.set_yscale("log")
ax_full.set_xlabel("epoch")
ax_full.set_ylabel("MSE loss (log scale)")
ax_full.set_title("All 30 epochs — same two rates diverge past 10³⁴", fontsize=10)
ax_full.grid(True, which="both", axis="y", color="#e1e0d9", linewidth=0.6)
ax_full.set_axisbelow(True)

fig.suptitle("Same gradient, five step sizes — gradual descent vs. overshoot", fontsize=12)
plt.tight_layout()
plt.savefig(f"{OUTPUT_DIR}/learning_rate_behavior.png", dpi=150)
plt.show()